## Introduction

Exploration of causal language models for model selection. These models are explained to be relatively simpler than conventional large language models (LLMs).

#### Installing the Required Libraries


In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA is available: {torch.cuda.is_available()}")

PyTorch version: 2.14.1+cpu
CUDA is available: False


In [ ]:
!pip install transformers datasets -q

In [ ]:
from datasets import load_dataset

# Retrieve the SQuAD dataset from Hugging Face
dataset = load_dataset("rajpurkar/squad")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [ ]:
def add_end_of_text(example):
    question = example["question"]
    example["question"] = f"{question}<|endoftext|>"
    return example


columns_to_remove = ["id", "title", "context", "answers"]

dataset = dataset.remove_columns(columns_to_remove)
dataset = dataset.map(add_end_of_text)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
# Display the first ten questions in the training split
train_questions = dataset["train"]["question"]

print(train_questions[:10])

['To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<|endoftext|>', 'What is in front of the Notre Dame Main Building?<|endoftext|>', 'The Basilica of the Sacred heart at Notre Dame is beside to which structure?<|endoftext|>', 'What is the Grotto at Notre Dame?<|endoftext|>', 'What sits on top of the Main Building at Notre Dame?<|endoftext|>', 'When did the Scholastic Magazine of Notre dame begin publishing?<|endoftext|>', "How often is Notre Dame's the Juggler published?<|endoftext|>", 'What is the daily student paper at Notre Dame called?<|endoftext|>', 'How many student news papers are found at Notre Dame?<|endoftext|>', 'In what year did the student paper Common Sense begin publication at Notre Dame?<|endoftext|>']


In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=model_checkpoint,
    use_fast=True
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
sequence = (
    "This tokenizer is being applied in CS197 at "
    "Harvard.<|endoftext|>"
)

tokens = tokenizer.tokenize(text=sequence)

print("Tokenized sequence:", tokens)

Tokenized sequence: ['This', 'Ġtoken', 'izer', 'Ġis', 'Ġbeing', 'Ġapplied', 'Ġin', 'ĠCS', '197', 'Ġat', 'ĠHarvard', '.', '<|endoftext|>']


In [ ]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Token IDs:", token_ids)

Token IDs: [1212, 11241, 7509, 318, 852, 5625, 287, 9429, 24991, 379, 11131, 13, 50256]


In [ ]:
# Examine the tokenizer's output for a sample sentence
sequence = (
    "This tokenizer is being applied in CS197 at "
    "Harvard.<|endoftext|>"
)

encoded_sequence = tokenizer(
    text=sequence,
    return_tensors=None
)

print(encoded_sequence)

{'input_ids': [1212, 11241, 7509, 318, 852, 5625, 287, 9429, 24991, 379, 11131, 13, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
def tokenize_function(examples):
    questions = examples["question"]

    tokenized_output = tokenizer(
        questions,
        truncation=True
    )

    return tokenized_output


tokenized_datasets = dataset.map(
    function=tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=["question"]
)

Map (num_proc=4):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
block_size = 128


def group_texts(examples):

    # Combine all token sequences into one long sequence per feature
    concatenated_examples = {
        key: [token for sequence in sequences for token in sequence]
        for key, sequences in examples.items()
    }

    # Determine the total number of tokens
    total_length = len(concatenated_examples["input_ids"])

    # Discard any remaining tokens that cannot fill a complete block
    total_length = (total_length // block_size) * block_size

    # Divide the concatenated sequences into fixed-size blocks
    result = {}

    for key, tokens in concatenated_examples.items():
        result[key] = [
            tokens[start:start + block_size]
            for start in range(0, total_length, block_size)
        ]

    # Use input_ids as the language modelling targets
    result["labels"] = [ids.copy() for ids in result["input_ids"]]

    return result


lm_datasets = tokenized_datasets.map(
    function=group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4
)

Map (num_proc=4):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
train_questions = dataset["train"]["question"]

for index, question in enumerate(train_questions[:10], start=1):
    print(f"{index}. {question}")

1. To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<|endoftext|>
2. What is in front of the Notre Dame Main Building?<|endoftext|>
3. The Basilica of the Sacred heart at Notre Dame is beside to which structure?<|endoftext|>
4. What is the Grotto at Notre Dame?<|endoftext|>
5. What sits on top of the Main Building at Notre Dame?<|endoftext|>
6. When did the Scholastic Magazine of Notre dame begin publishing?<|endoftext|>
7. How often is Notre Dame's the Juggler published?<|endoftext|>
8. What is the daily student paper at Notre Dame called?<|endoftext|>
9. How many student news papers are found at Notre Dame?<|endoftext|>
10. In what year did the student paper Common Sense begin publication at Notre Dame?<|endoftext|>


In [ ]:
first_block = lm_datasets["train"]["input_ids"][0]

print("First tokenized block:")
print(first_block)

First tokenized block:
[2514, 4150, 750, 262, 5283, 5335, 7910, 1656, 287, 1248, 3365, 287, 406, 454, 8906, 4881, 30, 50256, 2061, 318, 287, 2166, 286, 262, 23382, 20377, 8774, 11819, 30, 50256, 464, 32520, 3970, 286, 262, 17380, 2612, 379, 23382, 20377, 318, 13970, 284, 543, 4645, 30, 50256, 2061, 318, 262, 10299, 33955, 379, 23382, 20377, 30, 50256, 2061, 10718, 319, 1353, 286, 262, 8774, 11819, 379, 23382, 20377, 30, 50256, 2215, 750, 262, 3059, 349, 3477, 11175, 286, 23382, 288, 480, 2221, 12407, 30, 50256, 2437, 1690, 318, 23382, 20377, 338, 262, 39296, 1754, 3199, 30, 50256, 2061, 318, 262, 4445, 3710, 3348, 379, 23382, 20377, 1444, 30, 50256, 2437, 867, 3710, 1705, 9473, 389, 1043, 379, 23382, 20377, 30, 50256, 818, 644, 614, 750, 262, 3710, 3348]


In [ ]:
first_block = lm_datasets["train"]["input_ids"][0]

decoded_text = tokenizer.decode(first_block)

print(decoded_text)

To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<|endoftext|>What is in front of the Notre Dame Main Building?<|endoftext|>The Basilica of the Sacred heart at Notre Dame is beside to which structure?<|endoftext|>What is the Grotto at Notre Dame?<|endoftext|>What sits on top of the Main Building at Notre Dame?<|endoftext|>When did the Scholastic Magazine of Notre dame begin publishing?<|endoftext|>How often is Notre Dame's the Juggler published?<|endoftext|>What is the daily student paper at Notre Dame called?<|endoftext|>How many student news papers are found at Notre Dame?<|endoftext|>In what year did the student paper


In [ ]:
# Randomly shuffle both dataset splits
shuffled_train = lm_datasets["train"].shuffle(seed=42)
shuffled_eval = lm_datasets["validation"].shuffle(seed=42)

# Select 100 examples from each split
small_train_dataset = shuffled_train.select(range(100))
small_eval_dataset = shuffled_eval.select(range(100))

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_checkpoint
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
training_args = TrainingArguments(
    f"{model_checkpoint}-squad",
    #evaluation_strategy="epoch", #upgraded to the below argument
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.693164
2,No log,3.516703
3,No log,3.459380


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=39, training_loss=3.857268504607372, metrics={'train_runtime': 533.0793, 'train_samples_per_second': 0.563, 'train_steps_per_second': 0.073, 'total_flos': 9798628147200.0, 'train_loss': 3.857268504607372, 'epoch': 3.0})

In [ ]:
save_directory = "./gpt2-squad"

tokenizer.save_pretrained(save_directory)

print(f"Tokenizer saved to: {save_directory}")

Tokenizer saved to: ./gpt2-squad


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

local_dir = "./distilgpt2-squad/checkpoint-39"

# Restore the fine-tuned model
loaded_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=local_dir
)

# Restore the corresponding tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=local_dir
)

print("Successfully loaded the model and tokenizer.")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Successfully loaded the model and tokenizer.


In [ ]:
start_text = ("A speedrun is a playthrough of a video game, \
or section of a video game, with the goal of \
completing it as fast as possible. Speedruns \
often follow planned routes, which may incorporate sequence \
breaking, and might exploit glitches that allow sections to \
be skipped or completed more quickly than intended. ")

prompt = "What is the"
inputs = tokenizer(
     start_text + prompt,
     add_special_tokens=False,
     return_tensors="pt")["input_ids"]

prompt_length = len(tokenizer.decode(inputs[0]))
outputs = model.generate(
     inputs,
     max_length=100,
     do_sample=True,
     top_k=50,
     top_p=0.95,
     temperature=0.9,
     num_return_sequences=3)

generated = prompt + tokenizer.decode(outputs[0])[prompt_length + 1:]

print(tokenizer.decode(outputs[0]))

A speedrun is a playthrough of a video game, or section of a video game, with the goal of completing it as fast as possible. Speedruns often follow planned routes, which may incorporate sequence breaking, and might exploit glitches that allow sections to be skipped or completed more quickly than intended. What is the best way to earn credits for completing a video game?<|endoftext|>
